# Wan2GP on Google Colab

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Colab session. 

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. Most Wan2GP models exceed that budget; the Wan 2.2 TextImage2Video FastWan model works, producing roughly a 5 second 480p clip in about 8 minutes.

> **Tip:** lower the resolution in the Wan2GP interface before your first generation. The models default to 1280x720, which is more than a free Colab GPU handles comfortably.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path and optional persistent data storage


Set `USE_GOOGLE_DRIVE_DATA = True` if you want Google Drive to keep checkpoints, LoRAs, outputs and model caches across Colab restarts.


In [ ]:
from pathlib import Path


# CHANGE THIS TO TRUE IF YOU WANT TO USE GOOGLE DRIVE FOR DATA STORAGE (PERSISTENT ACROSS SESSIONS)
USE_GOOGLE_DRIVE_DATA = False




DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent data)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime disk (ephemeral data)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Data root: {WAN_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [ ]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

MANAGED_PATHS = {'ckpts', 'loras', 'outputs', 'ffmpeg_bins'}

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    untracked = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'ls-files', '--others', '--exclude-standard'],
        check=True,
        capture_output=True,
        text=True,
    )
    user_files = [
        line for line in untracked.stdout.splitlines()
        if line.split('/', 1)[0] not in MANAGED_PATHS
    ]
    if user_files:
        preview = ', '.join(user_files[:5]) + (' …' if len(user_files) > 5 else '')
        print(f'Repository has your own files in it ({preview}). Skipping the update to keep them.')
    else:
        print('Repository already exists. Updating to the latest version...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', '--', '.'], check=True)
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


## 4. Install system dependencies

Install shared libraries needed for video and audio processing, plus a compatible FFmpeg build when Colab's preinstalled version is too old. The FFmpeg download is integrity-checked and kept separate from system packages. If you see a warning about skipping an extra repository, it is safe to ignore.

In [ ]:
import hashlib, json, os, platform, re, shutil, subprocess, tarfile, tempfile, urllib.request
from pathlib import Path

APT_PACKAGES = ['libglib2.0-0', 'libgl1', 'libportaudio2']
FFMPEG_RELEASE_API = 'https://api.github.com/repos/BtbN/FFmpeg-Builds/releases/latest'
FFMPEG_DOWNLOAD_PREFIX = 'https://github.com/BtbN/FFmpeg-Builds/releases/download/'
FFMPEG_BIN_DIR = WAN2GP_ROOT / 'ffmpeg_bins'
FFMPEG_CACHE_DIR = WAN_CACHE_DIR / 'downloads'
FFMPEG_ARCHITECTURES = {
    'x86_64': 'linux64',
    'amd64': 'linux64',
    'aarch64': 'linuxarm64',
    'arm64': 'linuxarm64',
}

def dpkg_installed(package: str) -> bool:
    return subprocess.run(['dpkg', '-s', package], capture_output=True).returncode == 0

missing = [package for package in APT_PACKAGES if not dpkg_installed(package)]
if missing:
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    print('Installing:', ', '.join(missing))
    subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
    subprocess.run([
        'sudo', 'apt-get', 'install', '-y', '--no-install-recommends', *missing
    ], check=True, env=env)
else:
    print('System libraries already installed.')

def ffmpeg_supports_required_options(binary: Path) -> bool:
    try:
        result = subprocess.run(
            [str(binary), '-hide_banner', '-h', 'full'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=30,
        )
    except (OSError, subprocess.TimeoutExpired):
        return False
    option_pattern = re.compile(r'^\s*-fps_mode(?:\[:[^]]+\])?(?:\s|$)', re.MULTILINE)
    return result.returncode == 0 and option_pattern.search(result.stdout) is not None

def executable_works(binary: Path) -> bool:
    try:
        return subprocess.run(
            [str(binary), '-version'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=30,
        ).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def github_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={
            'Accept': 'application/vnd.github+json',
            'User-Agent': 'Wan2GP-Colab-notebook',
            'X-GitHub-Api-Version': '2022-11-28',
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)

def download_file(url: str, destination: Path, expected_size: int) -> None:
    request = urllib.request.Request(url, headers={'User-Agent': 'Wan2GP-Colab-notebook'})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open('wb') as output:
        downloaded = 0
        next_report = 10
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            downloaded += len(chunk)
            if expected_size:
                percent = downloaded * 100 // expected_size
                if percent >= next_report:
                    print(f'FFmpeg download: {min(percent, 100)}%')
                    next_report += 10
    if expected_size and downloaded != expected_size:
        raise RuntimeError(f'Incomplete FFmpeg download: expected {expected_size} bytes, got {downloaded}.')

def install_current_stable_ffmpeg() -> tuple[Path, Path]:
    machine = platform.machine().lower()
    asset_architecture = FFMPEG_ARCHITECTURES.get(machine)
    if asset_architecture is None:
        raise RuntimeError(f'No FFmpeg build is configured for the {machine} architecture.')
    asset_pattern = re.compile(
        rf'^ffmpeg-n(?P<version>\d+\.\d+)-latest-{asset_architecture}-gpl-(?P=version)\.tar\.xz$'
    )

    release = github_json(FFMPEG_RELEASE_API)
    candidates = []
    for asset in release.get('assets', []):
        match = asset_pattern.fullmatch(asset.get('name', ''))
        if match:
            version = tuple(int(part) for part in match.group('version').split('.'))
            candidates.append((version, asset))
    if not candidates:
        raise RuntimeError(f'No stable {asset_architecture} FFmpeg build was found in the upstream release.')

    _, asset = max(candidates, key=lambda item: item[0])
    asset_url = asset.get('browser_download_url', '')
    digest = asset.get('digest') or ''
    if not asset_url.startswith(FFMPEG_DOWNLOAD_PREFIX):
        raise RuntimeError(f'Unexpected FFmpeg download URL: {asset_url}')
    if not digest.startswith('sha256:'):
        raise RuntimeError('The FFmpeg release does not provide a SHA-256 digest; refusing an unverified download.')

    expected_sha256 = digest.split(':', 1)[1].lower()
    expected_size = int(asset.get('size') or 0)
    FFMPEG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = FFMPEG_CACHE_DIR / asset['name']
    archive_valid = (
        archive_path.is_file()
        and (not expected_size or archive_path.stat().st_size == expected_size)
        and sha256_file(archive_path) == expected_sha256
    )
    if archive_valid:
        print(f'Using verified cached FFmpeg archive: {archive_path.name}')
    else:
        print(f'Downloading stable FFmpeg build: {asset["name"]}')
        with tempfile.NamedTemporaryFile(dir=FFMPEG_CACHE_DIR, suffix='.download', delete=False) as temp_file:
            temp_path = Path(temp_file.name)
        try:
            download_file(asset_url, temp_path, expected_size)
            actual_sha256 = sha256_file(temp_path)
            if actual_sha256 != expected_sha256:
                raise RuntimeError(
                    f'FFmpeg checksum mismatch: expected {expected_sha256}, got {actual_sha256}.'
                )
            os.replace(temp_path, archive_path)
        finally:
            temp_path.unlink(missing_ok=True)

    FFMPEG_BIN_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode='r:xz') as archive:
        members = {}
        for member in archive.getmembers():
            member_path = Path(member.name)
            if member.isfile() and member_path.parent.name == 'bin' and member_path.name in {'ffmpeg', 'ffprobe'}:
                if member_path.name in members:
                    raise RuntimeError(f'Duplicate {member_path.name} in FFmpeg archive.')
                members[member_path.name] = member
        if set(members) != {'ffmpeg', 'ffprobe'}:
            raise RuntimeError('The FFmpeg archive did not contain both ffmpeg and ffprobe.')
        for binary_name, member in members.items():
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f'Could not read {binary_name} from the FFmpeg archive.')
            destination = FFMPEG_BIN_DIR / binary_name
            temporary_destination = FFMPEG_BIN_DIR / f'.{binary_name}.tmp'
            with source, temporary_destination.open('wb') as output:
                shutil.copyfileobj(source, output)
            temporary_destination.chmod(0o755)
            os.replace(temporary_destination, destination)

    return FFMPEG_BIN_DIR / 'ffmpeg', FFMPEG_BIN_DIR / 'ffprobe'

local_ffmpeg = FFMPEG_BIN_DIR / 'ffmpeg'
local_ffprobe = FFMPEG_BIN_DIR / 'ffprobe'
system_ffmpeg_name = shutil.which('ffmpeg')
system_ffprobe_name = shutil.which('ffprobe')
system_ffmpeg = Path(system_ffmpeg_name) if system_ffmpeg_name else None
system_ffprobe = Path(system_ffprobe_name) if system_ffprobe_name else None

if local_ffmpeg.exists() or local_ffprobe.exists():
    if ffmpeg_supports_required_options(local_ffmpeg) and executable_works(local_ffprobe):
        ffmpeg_binary, ffprobe_binary = local_ffmpeg, local_ffprobe
        print('Using the compatible FFmpeg build already installed for Wan2GP.')
    else:
        print('The Wan2GP FFmpeg build is incomplete or outdated; updating it...')
        ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()
elif (
    system_ffmpeg is not None
    and system_ffprobe is not None
    and ffmpeg_supports_required_options(system_ffmpeg)
    and executable_works(system_ffprobe)
):
    ffmpeg_binary, ffprobe_binary = system_ffmpeg, system_ffprobe
    print("Using Colab's compatible system FFmpeg build.")
else:
    print("Colab's FFmpeg is missing or does not support -fps_mode; installing a current stable build...")
    ffmpeg_binary, ffprobe_binary = install_current_stable_ffmpeg()

if not ffmpeg_supports_required_options(ffmpeg_binary) or not executable_works(ffprobe_binary):
    raise RuntimeError('FFmpeg verification failed after installation.')

os.environ['FFMPEG_BINARY'] = str(ffmpeg_binary.resolve())
os.environ['FFPROBE_BINARY'] = str(ffprobe_binary.resolve())
binary_dir = str(ffmpeg_binary.resolve().parent)
path_parts = [part for part in os.environ.get('PATH', '').split(os.pathsep) if part != binary_dir]
os.environ['PATH'] = os.pathsep.join([binary_dir, *path_parts])
version_line = subprocess.run(
    [os.environ['FFMPEG_BINARY'], '-version'], check=True, capture_output=True, text=True
).stdout.splitlines()[0]
print(f'FFmpeg ready: {version_line}')
print(f'FFmpeg binary: {os.environ["FFMPEG_BINARY"]}')


## 5. Install Python dependencies

Install PyTorch and Wan2GP's Python packages. This takes a few minutes.

In [ ]:
import json, os, subprocess, sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
REPLACEMENT_TORCH = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
UNSUPPORTED_TORCH = ('2.8.0', '2.9.')
ONNXRUNTIME_GPU = 'onnxruntime-gpu==1.22.0'

TORCH_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import importlib.metadata as md, torch\n'
    '    for name in ("torch", "torchvision", "torchaudio"):\n'
    '        try:\n'
    '            info[name] = md.version(name)\n'
    '        except Exception:\n'
    '            pass\n'
    '    info["cuda"] = torch.cuda.is_available()\n'
    '    if info["cuda"]:\n'
    '        info["device"] = torch.cuda.get_device_name(0)\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

ONNX_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import onnxruntime\n'
    '    info["version"] = onnxruntime.__version__\n'
    '    info["providers"] = onnxruntime.get_available_providers()\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

env = os.environ.copy()

if USE_GOOGLE_DRIVE_DATA:
    uv_cache_dir = WAN_CACHE_DIR / 'uv'
    uv_cache_dir.mkdir(parents=True, exist_ok=True)
    env['UV_CACHE_DIR'] = str(uv_cache_dir)
    env['UV_LINK_MODE'] = 'copy'

def uv_pip(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', *args], check=True, env=env)

def uv_install(*args):
    uv_pip('install', '--system', *args)

def probe(code):
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, env=env)
    try:
        return json.loads(result.stdout.strip().splitlines()[-1])
    except Exception:
        return {}

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True, env=env)

torch_version = probe(TORCH_PROBE).get('torch', '')
try:
    torch_release = tuple(int(part) for part in torch_version.split('.')[:2])
except ValueError:
    torch_release = ()
torch_usable = (
    torch_release >= (2, 7)
    and not torch_version.startswith(UNSUPPORTED_TORCH)
)

if torch_usable:
    print(f'Using the PyTorch already installed in this runtime ({torch_version}).')
else:
    print('Installing PyTorch...')
    uv_install(*REPLACEMENT_TORCH, '--index-url', PYTORCH_INDEX_URL)

print('Installing Wan2GP packages...')
uv_install('-r', str(WAN2GP_ROOT / 'requirements.txt'), '--index-strategy', 'unsafe-best-match')

uv_install(ONNXRUNTIME_GPU)
onnx = probe(ONNX_PROBE)
if not onnx.get('version'):
    uv_pip('uninstall', '--system', 'onnxruntime-gpu')
    uv_install('onnxruntime>=1.22.0')
    onnx = probe(ONNX_PROBE)

report = probe(TORCH_PROBE)
if not report.get('torch'):
    raise RuntimeError(
        'PyTorch is not working after installation. Open Runtime -> Restart session, then run Steps 2 to 5 again.'
    )
if not report.get('cuda'):
    raise RuntimeError(
        'No GPU detected. Open Runtime -> Change runtime type, choose GPU, then run Steps 1 and 5 again.'
    )

print(f"Ready: PyTorch {report['torch']} on {report['device']}.")
if 'CUDAExecutionProvider' not in onnx.get('providers', []):
    print('Note: background removal will run on the CPU.')


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 6 launches cleanly in Colab.


In [ ]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 6. UGC Ads Studio

Turn a video Wan2GP just generated into a publish-ready UGC-style ad: a bold hook for the first couple of seconds, burned-in captions for the rest of the script, and a crop to the aspect ratio your ad platform wants (9:16 for TikTok/Reels/Shorts, 4:5 or 1:1 for feed placements, 16:9 for YouTube).

This launches its own Gradio app on a separate public link (distinct from Wan2GP's) so both stay usable side by side. Generate a video with Wan2GP first (below), then come back to this tab, pick that clip from the dropdown (or upload any other video), add your hook and script, and click **Generate ad**. Outputs are saved under `outputs/ugc_ads` alongside Wan2GP's own outputs, so they persist to Google Drive too when persistent storage is enabled.


In [ ]:
import json
import os
import subprocess
import tempfile
import time
from pathlib import Path

import gradio as gr

ASPECT_RATIOS = {
    '9:16 (TikTok / Reels / Shorts)': (1080, 1920),
    '4:5 (Instagram feed)': (1080, 1350),
    '1:1 (Square)': (1080, 1080),
    '16:9 (YouTube / landscape)': (1920, 1080),
}
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.webm', '.mkv', '.avi'}
WORDS_PER_SECOND = 2.5
MIN_CHUNK_SECONDS = 0.9
MAX_WORDS_PER_CHUNK = 6

UGC_ADS_OUTPUT_DIR = (WAN_OUTPUTS_DIR / 'ugc_ads').resolve()
UGC_ADS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_bold_font():
    candidates = [
        Path('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'),
        Path('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    subprocess.run(
        ['sudo', 'apt-get', 'install', '-y', '--no-install-recommends', 'fonts-dejavu-core'],
        check=True, env=env,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise RuntimeError('No bold font found for UGC Ads Studio captions, even after installing fonts-dejavu-core.')


UGC_FONT_PATH = find_bold_font()


def list_wan2gp_outputs():
    if not WAN_OUTPUTS_DIR.exists():
        return []
    videos = [
        path for path in WAN_OUTPUTS_DIR.rglob('*')
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS and UGC_ADS_OUTPUT_DIR not in path.parents
    ]
    videos.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    return [str(path) for path in videos]


def probe_duration_seconds(input_video, ffprobe_binary):
    result = subprocess.run(
        [ffprobe_binary, '-v', 'error', '-show_entries', 'format=duration', '-of', 'json', str(input_video)],
        check=True, capture_output=True, text=True,
    )
    return float(json.loads(result.stdout)['format']['duration'])


def chunk_script(script_text, max_words=MAX_WORDS_PER_CHUNK):
    words = script_text.split()
    return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)] or []


def compute_caption_timings(chunks, start_at, end_at):
    if not chunks:
        return []
    window = max(end_at - start_at, 0.1)
    weights = [max(len(chunk.split()) / WORDS_PER_SECOND, MIN_CHUNK_SECONDS) for chunk in chunks]
    total_weight = sum(weights)
    scale = window / total_weight if total_weight > 0 else 0
    timings = []
    cursor = start_at
    for chunk, weight in zip(chunks, weights):
        duration = weight * scale
        timings.append((cursor, cursor + duration, chunk))
        cursor += duration
    return timings


def build_filtergraph(width, height, hook_text, caption_timings, hook_duration, font_path, brand_hex, work_dir):
    # Caption text is written to on-disk textfile= inputs rather than inlined
    # into the filter_complex string: this ffmpeg build's filtergraph parser
    # terminates a drawtext text= value at the first unescaped ':' regardless
    # of backslash-escaping or single-quoting, and drawtext's own %-expansion
    # aborts the whole draw silently on an unescaped '%'. Routing text through
    # files with expansion=none sidesteps both.
    work_dir = Path(work_dir)
    filters = [
        f'scale={width}:{height}:force_original_aspect_ratio=increase',
        f'crop={width}:{height}',
    ]

    def write_caption_file(name, text):
        path = work_dir / f'caption_{name}.txt'
        path.write_text(text, encoding='utf-8')
        return path

    def drawtext(caption_path, y_expr, start, end, fontsize, box_color):
        return (
            f'drawtext=fontfile={font_path}:textfile={caption_path}:expansion=none:'
            f'fontsize={fontsize}:fontcolor=white:borderw=3:bordercolor=black@0.8:'
            f'box=1:boxcolor={box_color}@0.55:boxborderw=18:'
            f'x=(w-text_w)/2:y={y_expr}:'
            f'line_spacing=6:'
            f"enable='between(t,{start:.3f},{end:.3f})'"
        )

    text_filters = []
    if hook_text.strip() and hook_duration > 0:
        hook_path = write_caption_file('hook', hook_text.strip())
        text_filters.append(drawtext(hook_path, 'h*0.12', 0, hook_duration, 64, brand_hex))
    for index, (start, end, chunk) in enumerate(caption_timings):
        caption_path = write_caption_file(index, chunk)
        text_filters.append(drawtext(caption_path, 'h*0.78', start, end, 46, 'black'))

    full_chain = ','.join(filters + text_filters) if text_filters else ','.join(filters)
    return f'[0:v]{full_chain}[outv]'


def render_ugc_ad(input_video, output_path, hook_text, script_text, aspect_ratio_label, hook_duration, brand_hex):
    ffmpeg_binary = os.environ.get('FFMPEG_BINARY', 'ffmpeg')
    ffprobe_binary = os.environ.get('FFPROBE_BINARY', 'ffprobe')
    width, height = ASPECT_RATIOS[aspect_ratio_label]
    duration = probe_duration_seconds(input_video, ffprobe_binary)
    hook_duration = min(hook_duration, duration)
    chunks = chunk_script(script_text)
    caption_timings = compute_caption_timings(chunks, hook_duration, duration)

    output_path = Path(output_path)
    with tempfile.TemporaryDirectory(prefix='ugc_ads_captions_') as work_dir:
        filtergraph = build_filtergraph(
            width, height, hook_text, caption_timings, hook_duration, UGC_FONT_PATH, brand_hex, work_dir
        )
        cmd = [
            ffmpeg_binary, '-y',
            '-i', str(input_video),
            '-filter_complex', filtergraph,
            '-map', '[outv]',
            '-map', '0:a?',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
            '-c:a', 'aac', '-shortest',
            str(output_path),
        ]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    return output_path


def generate_ad(source_path, uploaded_video, hook_text, hook_duration, script_text, aspect_ratio_label, brand_color):
    if isinstance(uploaded_video, str) or uploaded_video is None:
        uploaded_path = uploaded_video
    else:
        uploaded_path = getattr(uploaded_video, 'name', None) or getattr(uploaded_video, 'path', None)
    input_path = uploaded_path or source_path
    if not input_path or not Path(input_path).exists():
        return None, 'Pick a Wan2GP output from the dropdown or upload a video first.'
    if not hook_text.strip() and not script_text.strip():
        return None, 'Add a hook and/or a script — there is nothing to caption otherwise.'

    brand_hex = '0x' + brand_color.lstrip('#') if brand_color else '0x1DA1F2'
    timestamp = time.strftime('%Y%m%d-%H%M%S')
    output_path = UGC_ADS_OUTPUT_DIR / f'ugc_ad_{timestamp}.mp4'
    try:
        render_ugc_ad(input_path, output_path, hook_text, script_text, aspect_ratio_label, hook_duration, brand_hex)
    except subprocess.CalledProcessError as exc:
        detail = exc.stderr[-800:] if exc.stderr else str(exc)
        return None, f'ffmpeg failed:\n```\n{detail}\n```'
    return str(output_path), f'Saved to `{output_path}`'


with gr.Blocks(title='UGC Ads Studio') as ugc_ads_demo:
    gr.Markdown(
        '# UGC Ads Studio\n'
        'Hook + burned-in captions + ad aspect ratio, applied to a Wan2GP clip.'
    )
    with gr.Row():
        with gr.Column():
            refresh_button = gr.Button('🔄 Refresh Wan2GP outputs list')
            source_dropdown = gr.Dropdown(label='Pick a Wan2GP output', choices=list_wan2gp_outputs())
            uploaded_video = gr.Video(label='...or upload a video', sources=['upload'])
            hook_text = gr.Textbox(label='Hook (first few seconds, big bold text)', placeholder='STOP scrolling...')
            hook_duration = gr.Slider(0, 5, value=2.2, step=0.1, label='Hook duration (seconds)')
            script_text = gr.Textbox(
                label='Ad script / captions', lines=5,
                placeholder='Type the voiceover or on-screen script the rest of the ad should caption...',
            )
            aspect_ratio = gr.Dropdown(
                label='Aspect ratio', choices=list(ASPECT_RATIOS.keys()),
                value='9:16 (TikTok / Reels / Shorts)',
            )
            brand_color = gr.ColorPicker(label='Hook highlight color', value='#1DA1F2')
            generate_button = gr.Button('🎬 Generate ad', variant='primary')
        with gr.Column():
            output_video = gr.Video(label='Ad-ready output')
            status_text = gr.Markdown()

    refresh_button.click(fn=lambda: gr.update(choices=list_wan2gp_outputs()), outputs=source_dropdown)
    generate_button.click(
        fn=generate_ad,
        inputs=[source_dropdown, uploaded_video, hook_text, hook_duration, script_text, aspect_ratio, brand_color],
        outputs=[output_video, status_text],
    )

ugc_ads_demo.launch(server_port=7861, share=True, prevent_thread_lock=True)
print('UGC Ads Studio launched — use the public link above once Wan2GP (next cell) has produced a clip to work from.')


## 7. Launch Wan2GP

Run the Gradio interface. You will find the gradio link in the output. Click on the link to access the UI. Keep the cell running to stay connected; stop it with the square **Stop** button when you are finished.

In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')
cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', '5',
]

if USE_GOOGLE_DRIVE_DATA:
    print('Using Google Drive for checkpoints, LoRAs, outputs and caches.')
else:
    print('Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.')
print('Launching Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')
